# GAT with Status Embedding — Next-Event Prediction on BPI Challenge 2012

This notebook uses the **Dual-GAT with edge-type (status) embeddings** on the BPI 2012 dataset.

BPI 2012 is an ideal fit for this model because it has **diverse lifecycle transitions**
(`COMPLETE`, `SCHEDULE`, `START`, etc.) — the edge-type embedding can learn meaningful
transition-specific representations that improve prediction.

### Notebook outline
1. Load & Explore
2. Train
3. Evaluate
4. Summary

In [1]:
from pathlib import Path
import sys, warnings

root = Path.cwd().resolve()
src = root / "src" if (root / "src").exists() else root.parent / "src"
sys.path.insert(0, str(src))
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load & Explore the Dataset

In [2]:
import pandas as pd

data_dir = Path.cwd() / "data" if (Path.cwd() / "data").exists() else Path.cwd() / "examples" / "data"
df = pd.read_csv(data_dir / "BPI_Challenge_2012.csv")

df = df.rename(columns={
    "case:concept:name": "case_id",
    "concept:name":      "event",
    "time:timestamp":    "time",
    "lifecycle:transition": "status",
    "case:AMOUNT_REQ":      "amount_req",
})
df = df.dropna(subset=["case_id", "event", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
df = df[df.groupby("case_id")["case_id"].transform("size") >= 2].reset_index(drop=True)

n_status = df["status"].nunique()
print(f"Cases: {df['case_id'].nunique()}  |  Events: {len(df)}  |  Activities: {df['event'].nunique()}")
print(f"Lifecycle statuses ({n_status}): {df['status'].unique().tolist()}")
df.head()

Cases: 13087  |  Events: 262200  |  Activities: 24
Lifecycle statuses (3): ['COMPLETE', 'SCHEDULE', 'START']


,org:resource,status,event,time,case:REG_DATE,case_id,amount_req
0,112.0,COMPLETE,A_SUBMITTED,2011-10-01 00:38:44.546000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
1,112.0,COMPLETE,A_PARTLYSUBMITTED,2011-10-01 00:38:44.880000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
2,112.0,COMPLETE,A_PREACCEPTED,2011-10-01 00:39:37.906000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
3,112.0,SCHEDULE,W_Completeren aanvraag,2011-10-01 00:39:38.875000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000
4,NaN,START,W_Completeren aanvraag,2011-10-01 11:36:46.437000+00:00,2011-10-01 00:38:44.546000+00:00,173688,20000


## 2. Train the Model

The `status_col` parameter tells the recipe to build edge-type embeddings
from consecutive `(event+status)` transitions.

In [3]:
from timegnn import gat_status

result = gat_status(
    df,
    case_col="case_id",
    event_col="event",
    time_col="time",
    status_col="status",
    cat_event=["event"],
    num_event=[],
    seq_cols=["amount_req"],
    cat_seq=[],
    num_seq=["amount_req"],
    num_epochs=3,
    batch_size=16,
)

model = result["model"]
pd.DataFrame(result["history"])

,epoch,train_loss,train_acc,test_loss,test_acc
0,1,0.557370,0.809797,0.551113,0.802757
1,2,0.484835,0.828998,0.500159,0.843193
2,3,0.474067,0.832628,0.511490,0.835968


## 3. Evaluate with Comprehensive Metrics

In [4]:
import torch
from torch.utils.data import DataLoader
from timegnn import (
    EventLogTransformer, top_k_accuracy, predict, predict_per_sequence,
    average_bleu_score, compute_dls_and_exact_match, analyze_sequence_errors,
)
from timegnn.data.pyg import CustomDataset, custom_collate_fn

transformer = EventLogTransformer(
    case_col="case_id", event_col="event", time_col="time",
    status_col="status",
    cat_event=["event"], num_event=[],
    seq_cols=["amount_req"], cat_seq=[], num_seq=["amount_req"],
    mode="gat_status",
).fit(df)

transformed = transformer.transform(df)
dataset = CustomDataset(transformed.event_features, transformed.labels)
loader  = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

preds_flat, labels_flat, outputs = predict(model, loader, device)
print(f"Top-1 accuracy: {top_k_accuracy(outputs, labels_flat, k=1):.4f}")
print(f"Top-3 accuracy: {top_k_accuracy(outputs, labels_flat, k=3):.4f}")
print(f"Top-5 accuracy: {top_k_accuracy(outputs, labels_flat, k=5):.4f}")

preds_seq, labels_seq = predict_per_sequence(model, loader, device)
bleu = average_bleu_score(preds_seq, labels_seq)
dls, exact = compute_dls_and_exact_match(preds_seq, labels_seq)
print(f"\nBLEU:  {bleu:.4f}")
print(f"DLS:   {dls:.4f}")
print(f"Exact: {exact:.4f}")

print("\nSequence error summary:")
print(analyze_sequence_errors(model, loader, device))

Top-1 accuracy: 0.8321
Top-3 accuracy: 0.9773
Top-5 accuracy: 0.9915

BLEU:  0.5301
DLS:   0.8044
Exact: 0.1360

Sequence error summary:
(array([   0, 8334,    0, 2736, 2061, 1663, 1069, 1466,  874,  829,  522,
        513,  871,  364,  871,  411,  909,  743,  849,  730,  927,  797,
        822,  748,  784,  785,  703,  662,  632,  628,  559,  548,  515,
        504,  488,  467,  429,  401,  392,  385,  342,  352,  329,  302,
        279,  279,  262,  227,  219,  217,  190,  193,  170,  165,  145,
        114,  131,  120,  126,   87,  104,   87,   82,   81,   85,   85,
         83,   71,   64,   61,   59,   55,   66,   59,   45,   43,   41,
         42,   39,   40,   27,   26,   27,   24,   19,   19,   17,   19,
         17,    8,   10,   18,    7,   13,   13,   17,   10,   11,   11,
          5,    5,    9,    8,    4,    7,   11,    8,    6,    7,    6,
          5,    6,    7,    8,    5,    5,    3,    2,    2,    4,    3,
          4,    5,    5,    4,    1,    0,    2,    3,    1

## 4. Summary

With BPI 2012's diverse lifecycle transitions, the edge-type embedding can learn
that transitions like `A_SUBMITTED→SCHEDULE` carry different predictive weight
than `W_Nabellen→COMPLETE`.

For even more expressiveness, combine edge-type embeddings with time decay
→ see **`gat_time_decay_status_bpi_full`**.